# Bootcamp Day 3 — Homework (scikit-learn)

**After Day 3 class.** Real datasets: Iris, Wine, Titanic-like.

Vocab allowed: `train_test_split`, `fit`, `predict`, `score`, `accuracy_score`, `classification_report`, `confusion_matrix`, `cross_val_score`, `LogisticRegression`, `RandomForestClassifier`, `RandomForestRegressor`.

The mantra: **split → fit → predict → score.** Always `random_state=42`.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/bootcamp_day3_homework.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets        import load_iris, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import accuracy_score, classification_report, confusion_matrix
print("ready")

---
## Problem 1 — Wine classifier baseline

Wine dataset (sklearn built-in) — 178 samples, 13 features, 3 classes.

1. Load with `load_wine(return_X_y=True)`
2. Split 80/20 with `random_state=42`, `stratify=y`
3. Fit `LogisticRegression(max_iter=5000)`
4. Report `accuracy` on test set

In [ ]:
# TODO
X, y = ...
X_tr, X_te, y_tr, y_te = ...

model    = ...
...                       # fit
acc      = ...

print("Wine LogReg accuracy:", round(acc, 3))

In [ ]:
assert acc > 0.9
print("Q1 ok")

<details><summary>Hint</summary>

```python
X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
acc = model.score(X_te, y_te)
```

</details>

---
## Problem 2 — confusion matrix on Iris

Train RandomForest on Iris. Compute and print the confusion matrix.

1. Use `random_state=42` everywhere
2. `cm` — 3×3 confusion matrix array
3. Verify: total of cm = test size

In [ ]:
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO
model  = ...
...                        # fit
y_pred = ...
cm     = ...

print("confusion matrix:\n", cm)
print("sum:", cm.sum(), "test size:", len(y_te))

In [ ]:
assert cm.shape == (3, 3)
assert cm.sum() == len(y_te)
print("Q2 ok")

<details><summary>Hint</summary>

```python
model = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
y_pred = model.predict(X_te)
cm = confusion_matrix(y_te, y_pred)
```

</details>

---
## Problem 3 — cross-validation comparison

Compare LogReg vs RandomForest on Iris using 5-fold CV.

1. `logreg_scores` — 5 CV scores for `LogisticRegression(max_iter=1000)`
2. `rf_scores` — 5 CV scores for `RandomForestClassifier(random_state=42)`
3. `winner` — string `"LogReg"` or `"RF"` based on higher mean
4. `mean_gap` — abs difference between the two means

In [ ]:
X, y = load_iris(return_X_y=True)

# TODO
logreg_scores = ...
rf_scores     = ...
winner        = ...
mean_gap      = ...

print(f"LogReg: {logreg_scores.mean():.3f} ± {logreg_scores.std():.3f}")
print(f"RF    : {rf_scores.mean():.3f} ± {rf_scores.std():.3f}")
print("winner:", winner, "gap:", round(mean_gap, 4))

In [ ]:
assert logreg_scores.shape == (5,)
assert rf_scores.shape == (5,)
assert winner in ("LogReg", "RF")
assert mean_gap >= 0
print("Q3 ok")

<details><summary>Hint</summary>

```python
logreg_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
rf_scores     = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)
winner = "LogReg" if logreg_scores.mean() > rf_scores.mean() else "RF"
mean_gap = abs(logreg_scores.mean() - rf_scores.mean())
```

</details>

---
## Problem 4 — predict on imbalanced classes

Synthetic dataset with imbalanced classes (95% class 0, 5% class 1). Build a classifier and report what really matters.

1. Use `train_test_split(stratify=y, random_state=42)`
2. Fit RandomForest
3. `acc` — accuracy
4. `class1_recall` — recall on the rare class 1 (read from classification_report dict)
5. Decide: is the model actually useful? `useful` = bool

In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, weights=[0.95, 0.05], random_state=0, n_features=10)
print("class balance:", np.bincount(y))

# TODO
X_tr, X_te, y_tr, y_te = ...
model = ...
...                        # fit
y_pred = ...
acc = ...

report_dict = classification_report(y_te, y_pred, output_dict=True)
class1_recall = ...     # read from report_dict — recall of class "1"

print(f"accuracy: {acc:.3f}   class1_recall: {class1_recall:.3f}")

# decide: model is useful only if it catches at least 50% of class 1
useful = ...

In [ ]:
assert 0 <= acc <= 1
assert 0 <= class1_recall <= 1
assert isinstance(useful, (bool, np.bool_))
print("Q4 ok — accuracy", round(acc,3), "recall on rare class:", round(class1_recall, 3))

<details><summary>Hint</summary>

```python
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
y_pred = model.predict(X_te)
acc = accuracy_score(y_te, y_pred)
report_dict = classification_report(y_te, y_pred, output_dict=True)
class1_recall = report_dict["1"]["recall"]
useful = class1_recall >= 0.5
```

For imbalanced data, accuracy lies. **Recall on the rare class** is the real metric.

</details>

---
## Problem 5 — capstone: Titanic end-to-end

Synthetic Titanic data. Build the full pipeline from raw data to a defended F1 score.

Required steps:
1. Fill `Age` NaN with median; fill `Embarked` NaN with `"S"`
2. Drop `Name`, `Ticket`
3. Encode `Sex` and `Embarked` with `get_dummies`
4. Split 80/20 stratified, `random_state=42`
5. Fit BOTH `LogisticRegression` and `RandomForestClassifier`
6. Print accuracy + classification_report for each
7. 5-fold cross-validation on the winning model
8. Compute `defended_score` = "F1 mean from CV, on the better model"

In [ ]:
from sklearn.metrics import f1_score
rng = np.random.default_rng(0)
n = 500
df = pd.DataFrame({
    "Name":     [f"P{i}" for i in range(n)],
    "Ticket":   rng.choice(["A123","B456","C789"], size=n),
    "Pclass":   rng.choice([1,2,3], size=n, p=[0.2, 0.3, 0.5]),
    "Sex":      rng.choice(["male","female"], size=n),
    "Age":      np.where(rng.random(n)>0.15, rng.integers(1,80,n).astype(float), np.nan),
    "SibSp":    rng.integers(0,5,n),
    "Fare":     rng.uniform(5,150,n),
    "Embarked": np.where(rng.random(n)>0.05, rng.choice(["S","C","Q"], size=n), None),
    "Survived": rng.integers(0,2,n),
})

# TODO — full pipeline

# 1-3: clean
df["Age"]      = ...
df["Embarked"] = ...
df = df.drop(columns=...)
df = pd.get_dummies(df, columns=...)

# 4: split
X = df.drop(columns=["Survived"])
y = df["Survived"]
X_tr, X_te, y_tr, y_te = ...

# 5: fit two models
logreg = ...
rf     = ...

# 6: report
for name, m in [("LogReg", logreg), ("RF", rf)]:
    p = m.predict(X_te)
    print(name, "acc:", round(accuracy_score(y_te, p), 3))
    print(classification_report(y_te, p))

# 7-8: CV on better model
better_model = ...    # pick based on accuracy
cv_f1 = cross_val_score(better_model, X, y, cv=5, scoring="f1")
defended_score = ...  # mean F1

In [ ]:
assert df["Age"].isna().sum() == 0
assert "Sex" not in df.columns, "didn't encode Sex"
assert "Name" not in df.columns
assert 0 <= defended_score <= 1
print("Q5 ok — capstone passed. Defended F1:", round(defended_score, 3))

<details><summary>Hint</summary>

```python
# clean
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna("S")
df = df.drop(columns=["Name", "Ticket"])
df = pd.get_dummies(df, columns=["Sex", "Embarked"])

# split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# fit two
logreg = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
rf     = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)

# CV on the winner
better_model = rf if rf.score(X_te, y_te) > logreg.score(X_te, y_te) else logreg
cv_f1 = cross_val_score(better_model, X, y, cv=5, scoring="f1")
defended_score = cv_f1.mean()
```

</details>

---
## Solutions

<details><summary>Show all</summary>

```python
# Q1 — Wine LogReg
X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
acc   = model.score(X_te, y_te)

# Q2 — Iris confusion matrix
model  = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
y_pred = model.predict(X_te)
cm     = confusion_matrix(y_te, y_pred)

# Q3 — CV comparison
logreg_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
rf_scores     = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)
winner   = "LogReg" if logreg_scores.mean() > rf_scores.mean() else "RF"
mean_gap = abs(logreg_scores.mean() - rf_scores.mean())

# Q4 — imbalanced classes
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model  = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
y_pred = model.predict(X_te)
acc           = accuracy_score(y_te, y_pred)
report_dict   = classification_report(y_te, y_pred, output_dict=True)
class1_recall = report_dict["1"]["recall"]
useful        = class1_recall >= 0.5

# Q5 — Titanic capstone
df["Age"]      = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna("S")
df = df.drop(columns=["Name", "Ticket"])
df = pd.get_dummies(df, columns=["Sex", "Embarked"])

X = df.drop(columns=["Survived"])
y = df["Survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

logreg = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
rf     = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)

for name, m in [("LogReg", logreg), ("RF", rf)]:
    p = m.predict(X_te)
    print(name, "acc:", round(accuracy_score(y_te, p), 3))
    print(classification_report(y_te, p))

better_model   = rf if rf.score(X_te, y_te) > logreg.score(X_te, y_te) else logreg
cv_f1          = cross_val_score(better_model, X, y, cv=5, scoring="f1")
defended_score = cv_f1.mean()
```

</details>